In [12]:
import sys
sys.path.append("..")
from src import index
from src import image_analysis
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import feature_importance

ImportError: cannot import name 'feature_importance' from 'sklearn.model_selection' (/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/__init__.py)

### vector X 2D (n_images, n_features) and vector Y 1D (n_images).

In [ ]:
X1 = index.get_data("../data/processed_images_Pepper/Bacterial_Spot","Bacterial_Spot") + index.get_data("../data/processed_images_Pepper/Healthy","Healthy")

Y = [x[0] for x in X1]        

In [ ]:
features = []
labels   = []
for path, label in X1:
    image = cv2.imread(path)

    if image is None:   
        raise ValueError(f"Image at path {path} could not be loaded.")
    
    hist = image_analysis.get_histogram(image)
    glcm = image_analysis.GLCM(image) 
    combined = np.concatenate([hist, glcm])
    features.append(combined)
    labels.append(label)

X = np.array(features)   # shape (n_images, n_features)
y = np.array(labels)     # shape (n_images,)

print(X.shape, y.shape)

(2475, 184) (2475,)


## Machine Learning

In [ ]:
liste = [10, 23, 35, 42, 59]

for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=liste[i])

    scaler = StandardScaler()

    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test  = scaler.transform(X_test)

    clf = SVC(kernel="rbf")

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

                precision    recall  f1-score   support

Bacterial_Spot       0.97      0.98      0.98       199
       Healthy       0.99      0.98      0.98       296

      accuracy                           0.98       495
     macro avg       0.98      0.98      0.98       495
  weighted avg       0.98      0.98      0.98       495

[[195   4]
 [  5 291]]
                precision    recall  f1-score   support

Bacterial_Spot       0.97      0.95      0.96       199
       Healthy       0.97      0.98      0.98       296

      accuracy                           0.97       495
     macro avg       0.97      0.97      0.97       495
  weighted avg       0.97      0.97      0.97       495

[[190   9]
 [  5 291]]
                precision    recall  f1-score   support

Bacterial_Spot       0.98      0.98      0.98       199
       Healthy       0.99      0.99      0.99       296

      accuracy                           0.99       495
     macro avg       0.99      0.98      0.99     

In [ ]:
liste = [10, 23, 35, 42, 59]

for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=liste[i])

    rdm_f = RandomForestClassifier(n_estimators=100, random_state=42)

    rdm_f.fit(X_train, y_train)

    y_pred = rdm_f.predict(X_test)
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

                precision    recall  f1-score   support

Bacterial_Spot       0.96      0.96      0.96       199
       Healthy       0.97      0.97      0.97       296

      accuracy                           0.97       495
     macro avg       0.97      0.97      0.97       495
  weighted avg       0.97      0.97      0.97       495

[[191   8]
 [  8 288]]
                precision    recall  f1-score   support

Bacterial_Spot       0.97      0.96      0.97       199
       Healthy       0.98      0.98      0.98       296

      accuracy                           0.97       495
     macro avg       0.97      0.97      0.97       495
  weighted avg       0.97      0.97      0.97       495

[[192   7]
 [  6 290]]
                precision    recall  f1-score   support

Bacterial_Spot       0.97      0.95      0.96       199
       Healthy       0.97      0.98      0.97       296

      accuracy                           0.97       495
     macro avg       0.97      0.96      0.97     

In [14]:
feature_importances = rdm_f.feature_importances_
print("Feature importances:", feature_importances)

print("texture (last 5):", feature_importances[-5:].sum())
print("color (first 179):", feature_importances[:179].sum())

Feature importances: [0.00244993 0.00614744 0.00909913 0.01727062 0.00572621 0.01589437
 0.00608214 0.00394253 0.00076839 0.00120966 0.00531691 0.00475803
 0.00263609 0.00526416 0.00685863 0.00991842 0.02401008 0.01406674
 0.00848246 0.02684608 0.01072893 0.0185846  0.01759817 0.0394386
 0.01907723 0.03672779 0.06826095 0.05938982 0.0468739  0.04920932
 0.03094201 0.04758999 0.02069469 0.02342269 0.02041026 0.01232737
 0.01315909 0.00558322 0.00396329 0.00514814 0.00198107 0.00727556
 0.00333312 0.00415188 0.01104939 0.0106882  0.00961959 0.0031462
 0.00140888 0.00150567 0.00150873 0.00082157 0.00098189 0.0010501
 0.00135801 0.00177813 0.00138171 0.00061494 0.00129236 0.00121317
 0.00105127 0.00142304 0.00059826 0.00071524 0.00050153 0.00104573
 0.00063246 0.00051791 0.00072527 0.00067977 0.00092373 0.00066467
 0.00076316 0.00061379 0.00064302 0.00028652 0.00063639 0.00091887
 0.00089642 0.00051785 0.0007183  0.00167096 0.00075781 0.00148348
 0.00123213 0.00048686 0.00130464 0.00228235

Texture is not worth the complexity : only 1.4% of the signal is in the texture features. So we can use only histogram color features for the classification.